In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')
%matplotlib inline
plt.style.use('ggplot')

In [2]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline, make_pipeline
from scipy.stats import skew
from sklearn.decomposition import PCA, KernelPCA
from xgboost import XGBRegressor

In [3]:
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR, LinearSVR
from sklearn.linear_model import ElasticNet, SGDRegressor, BayesianRidge
from sklearn.kernel_ridge import KernelRidge
from xgboost import XGBRegressor

In [4]:
pd.set_option('display.max_columns',500)
pd.set_option('display.max_rows',1000)

In [5]:
train=pd.read_csv('./X_train.csv', index_col=0)
test=pd.read_csv('./X_test.csv', index_col=0)
price_res=pd.read_csv('./y_train.csv', index_col=0)
print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print(f"price shape: {price_res.shape}")

train shape: (9460, 41)
test shape: (2366, 41)
price shape: (9460, 1)


# pipeline

In [184]:
import pandas as pd
from datetime import datetime
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, LabelEncoder,OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin

class DropColumns(BaseEstimator, TransformerMixin):
    def __init__(self, columns):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns)

class DateTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df = X.copy()
        df['建築完成年月'] = pd.to_datetime(df['建築完成年月'])
        today = datetime.today()
        df['date_str'] = df['交易年'].astype(str) + '-' + df['交易月'].astype(str).str.zfill(2) + '-' + df['交易日'].astype(str).str.zfill(2)
        df['交易日期'] = pd.to_datetime(df['date_str'])
        df['建築年紀'] = df['交易日期'].dt.year - df['建築完成年月'].dt.year
        df = df.drop(['建築完成年月', 'date_str','交易日期'], axis=1)
        
        return df

class AddShoppingPlaceFeature(BaseEstimator, TransformerMixin): 
    def fit(self, X, y=None): 
        return self 
    def transform(self, X): 
        X = X.copy() 
        X['購物場所'] = X[['大賣場', '超市', '百貨公司']].mean(axis=1) 
        return X 
class AddSchoolFeature(BaseEstimator, TransformerMixin): 
        def fit(self, X, y=None):
            return self 
        def transform(self, X): 
            X = X.copy() 
            X['學校'] = X[['托兒所', '國中', '高中職', '大學']].mean(axis=1) 
            return X
class AddGovernmentFeature(BaseEstimator, TransformerMixin): 
        def fit(self, X, y=None):
            return self 
        def transform(self, X): 
            X = X.copy() 
            X['政府'] = X[['警察局', '消防局']].mean(axis=1) 
            return X

class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.label_encoders = {}

    def fit(self, X, y=None):
        for col in X.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            le.fit(X[col])
            self.label_encoders[col] = le
        return self

    def transform(self, X):
        df = X.copy()
        for col, le in self.label_encoders.items():
            df[col] = le.transform(df[col])
        return df

class OneHotEncoderTransformer(BaseEstimator,  
 TransformerMixin):
    def __init__(self, handle_unknown='ignore'):
        """
        Initializes the OneHotEncoderTransformer.

        Args:
            handle_unknown (str, default='ignore'): Specifies how to handle
                unknown categories during transformation. Possible values are:
                - 'error': Raise an error for unknown categories.
                - 'ignore': Ignore unknown categories and treat them as
                  previously unseen categories.
        """
        self.encoders = {}
        self.handle_unknown = handle_unknown

    def fit(self, X, y=None):
        """
        Fits the transformer to the data X.

        Args:
            X (pd.DataFrame): The data to fit the transformer on.
            y (None, optional): Not used in this context.

        Returns:
            OneHotEncoderTransformer: The fitted transformer.
        """
        self.numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
        for col in X.select_dtypes(include=['object']).columns:
            #if col not in self.numerical_cols:
                encoder = OneHotEncoder(handle_unknown=self.handle_unknown)
                encoder.fit(X[[col]])  # Reshape to 2D for OneHotEncoder
                self.encoders[col] = encoder
        return self

    def transform(self, X):
        """
        Transforms the data X using one-hot encoding.

        Args:
            X (pd.DataFrame): The data to transform.

        Returns:
            pd.DataFrame: The transformed data with one-hot encoded columns 
                         and original numerical columns.
        """
        encoded_df = X[self.numerical_cols].copy()
        for col, encoder in self.encoders.items():
            encoded_features = encoder.transform(X[[col]]).toarray()
            encoded_df = pd.concat([encoded_df, pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out([col]))], axis=1)
        return encoded_df
class GaussianBasisFunctionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, ngrid=3):
        self.ngrid = ngrid

    def fit(self, X, y=None):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        xmin, xmax = min(coord_x), max(coord_x) 
        ymin, ymax = min(coord_y), max(coord_y) 
        self.xgrids = np.linspace(xmin, xmax, self.ngrid + 1)
        self.ygrids = np.linspace(ymin, ymax, self.ngrid + 1)

        self.mu_x_all, self.mu_y_all, self.std_x_all, self.std_y_all, self.count_all = [], [], [], [], []

        for i in range(self.ngrid):
            for j in range(self.ngrid):
                x1, x2 = self.xgrids[i], self.xgrids[i + 1]
                y1, y2 = self.ygrids[j], self.ygrids[j + 1]
                tmpindx = (x1 <= coord_x) * (coord_x < x2)
                tmpindy = (y1 <= coord_y) * (coord_y < y2)
                tmpind = tmpindx * tmpindy
                npoints = np.sum(tmpind)

                if npoints >= 20:
                    mu_x = np.mean(coord_x[tmpind])
                    mu_y = np.mean(coord_y[tmpind])
                    std_x = np.std(coord_x[tmpind])
                    std_y = np.std(coord_y[tmpind])

                    self.mu_x_all.append(mu_x)
                    self.mu_y_all.append(mu_y)
                    self.std_x_all.append(std_x)
                    self.std_y_all.append(std_y)
                    self.count_all.append(npoints)

        return self

    def transform(self, X):
        coord_x = X['橫坐標'].values
        coord_y = X['縱坐標'].values

        ngf = len(self.mu_x_all)
        gf_all = np.zeros((coord_x.shape[0], ngf))

        for ii in range(ngf):
            mu_x = self.mu_x_all[ii]
            mu_y = self.mu_y_all[ii]
            std_x = self.std_x_all[ii]
            std_y = self.std_y_all[ii]

            tmpgf = np.exp(-(coord_x - mu_x) ** 2 / (2 * std_x ** 2) - (coord_y - mu_y) ** 2 / (2 * std_y ** 2))
            gf_all[:, ii] = tmpgf

        gf_df = pd.DataFrame(gf_all, columns=[f"gf_{i}" for i in range(gf_all.shape[1])])
        
        return pd.concat([X, gf_df], axis=1)

# Create the pipeline
pipeline = Pipeline([
    ('date_transformer', DateTransformer()),
   #('add_shopping_place', AddShoppingPlaceFeature()), 
  # ('add_school', AddSchoolFeature()),
  #('add_GovernmentFeature', AddGovernmentFeature()),
 ('drop_unwanted_columns', DropColumns(columns=['托兒所', '國中', '高中職', '大學', '大賣場', '超市', '百貨公司'])),
# ('label_encoder', LabelEncoderTransformer()),
 ('onehot encoder',OneHotEncoderTransformer()),
   ('gaussian_basis', GaussianBasisFunctionTransformer(ngrid=3)),
  


])




In [185]:
# Apply the pipeline to your data
full = train.copy(True)

full_transformed = pipeline.fit_transform(full)
 


In [186]:
full_transformed.shape

(9460, 723)

# feature select

## corr 

In [187]:
import pandas as pd

def select_features(df1=price_res, df2=full_transformed
                    , threshold=0.01):
    """
    Selects features from df2 based on correlation with a target column in df1.

    Args:
        df1: DataFrame containing the target column.
        df2: DataFrame containing potential features.
        threshold: Correlation threshold for feature selection.

    Returns:
        DataFrame: Filtered DataFrame with selected features.
    """

    train_corr = df2.corrwith(df1['單價元平方公尺'], axis=0)
    train_corr = abs(train_corr)

    selected_features = train_corr[train_corr > threshold].index
    
    return selected_features

In [188]:
selected_fetures=select_features()
print(selected_fetures.shape)
full_selected=full_transformed[selected_fetures]

(516,)


In [189]:
# Apply the pipeline to your data
y_test = test.copy()
test_transformed = pipeline.fit_transform(y_test)

missing_cols = set(full_transformed.columns) - set(test_transformed.columns)
for col in missing_cols:
   test_transformed[col] = 0
test_selected=test_transformed[selected_fetures]


In [190]:
len(full_selected.columns),len(test_selected.columns)

(516, 516)

# model & Evaluate

In [176]:
from sklearn .model_selection import KFold

## score

In [157]:
# define cross validation strategy
def rmse_cv(model,X,y):
    
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kfold))
    return rmse


In [158]:
class grid():
    def __init__(self,model):
        self.model = model
    
    def grid_get(self,X,y,param_grid):
        grid_search = GridSearchCV(self.model,param_grid,cv=5, scoring="neg_mean_squared_error")
        grid_search.fit(X,y)
        print(grid_search.best_params_, np.sqrt(-grid_search.best_score_))
        grid_search.cv_results_['mean_test_score'] = np.sqrt(-grid_search.cv_results_['mean_test_score'])
        print(pd.DataFrame(grid_search.cv_results_)[['params','mean_test_score','std_test_score']])

### parameter
- xgb
  - 'colsample_bytree': 0.6, 'learning_rate': 0.1, 'max_depth': 8, 'min_child_weight': 1, 'subsample': 1.0
- rf
  - 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 20
- extra
  - 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300

In [196]:
xgb=XGBRegressor( learning_rate= 0.05, max_depth= 4, min_child_weight= 1, subsample= 1.0, n_estimators= 2000)
rf=RandomForestRegressor(max_depth= None, max_features= 'sqrt', min_samples_leaf= 1, min_samples_split= 2, n_estimators= 20,n_jobs=-1)
extra=ExtraTreesRegressor(max_depth= None, min_samples_leaf= 2, min_samples_split= 2, n_estimators= 300,n_jobs=-1)
rg = Ridge(alpha=0.1, random_state=1001)
rf2 = RandomForestRegressor(n_estimators=300, min_samples_split=3, random_state=1001, n_jobs=-1)
gbr = GradientBoostingRegressor()

# stacking


In [178]:
from sklearn.ensemble import StackingRegressor 

In [179]:
stack = StackingRegressor(
    estimators=[
        ('extra', extra),
        ('rf', rf),
        ('xgb',xgb),
      
    ],
    final_estimator=rg
)

In [198]:
stack = StackingRegressor(
    estimators=[
        ('extra', extra),
        ('rf', rf2),
        ('xgb',xgb),
      
    ],
    final_estimator=rg
)

In [181]:
from sklearn.metrics import root_mean_squared_error
def training(X_train: pd.DataFrame, y_train: pd.DataFrame, model):
    # K-fold cross validation
    kf = KFold(n_splits=5, shuffle=True, random_state=1001)
    rmse_list = []

    for index, (train_indexes, valid_indexes) in enumerate(kf.split(X_train)):
        print(f"Fold: {index+1}")
        X_train_kf, y_train_kf = X_train.iloc[train_indexes], y_train.iloc[train_indexes]
        X_valid_kf, y_valid_kf = X_train.iloc[valid_indexes], y_train.iloc[valid_indexes]
          
        # Training
        model.fit(X_train_kf, y_train_kf)
        y_pred = model.predict(X_valid_kf)

        rmse = root_mean_squared_error(y_valid_kf, y_pred.reshape(-1, 1))
        rmse_list.append(rmse)    
        print(f"RMSE: {rmse}\n")

    return np.mean(rmse_list)

X_train = full_select.copy(deep=True)
y_train = price_res.copy(deep=True)

rmse = training(X_train, y_train, stack)
print(f'Mean RMSE: {rmse}')

Fold: 1
RMSE: 33934.49287186073

Fold: 2
RMSE: 32379.945609244358

Fold: 3
RMSE: 31794.941548107738

Fold: 4
RMSE: 32667.758129103557

Fold: 5
RMSE: 32151.819355969936

Mean RMSE: 32585.791502857268


In [195]:
from sklearn.metrics import root_mean_squared_error
def training(X_train: pd.DataFrame, y_train: pd.DataFrame, model):
    # K-fold cross validation
    kf = KFold(n_splits=5, shuffle=True, random_state=1001)
    rmse_list = []

    for index, (train_indexes, valid_indexes) in enumerate(kf.split(X_train)):
        print(f"Fold: {index+1}")
        X_train_kf, y_train_kf = X_train.iloc[train_indexes], y_train.iloc[train_indexes]
        X_valid_kf, y_valid_kf = X_train.iloc[valid_indexes], y_train.iloc[valid_indexes]
          
        # Training
        model.fit(X_train_kf, y_train_kf)
        y_pred = model.predict(X_valid_kf)

        rmse = root_mean_squared_error(y_valid_kf, y_pred.reshape(-1, 1))
        rmse_list.append(rmse)    
        print(f"RMSE: {rmse}\n")

    return np.mean(rmse_list)

X_train = full_selected.copy(deep=True)
y_train = price_res.copy(deep=True)

rmse = training(X_train, y_train, model)
print(f'Mean RMSE: {rmse}')

Fold: 1
RMSE: 33901.41774317198

Fold: 2
RMSE: 32397.164504248904

Fold: 3
RMSE: 31757.70991009324

Fold: 4
RMSE: 32641.93103769933

Fold: 5
RMSE: 32204.12716130596

Mean RMSE: 32580.47007130388


In [ ]:
score =rmse_cv(stack,full_selected,price_res['單價元平方公尺']),  rmse_cv(stack,full_selected,price_res['單價元平方公尺']).mean()
score

In [ ]:
stack.fit(full_selected,price_res['單價元平方公尺'])

In [ ]:
res=stack.predict(test_selected)


 
# 建立 DataFrame
df = pd.DataFrame({'單價元平方公尺': res})

# 新增 Id 欄位
# Create a DataFrame, prioritizing 'Id'
df = pd.DataFrame({'Id': df.index, '單價元平方公尺': res})

# Specify the output directory
output_dir = './output'  # Replace with your desired path
output_file = output_dir + '/output.csv'



# # 將 DataFrame 輸出為 CSV 檔案
df.to_csv(output_file, index=False)